# 01 · EDA

Smart MCQ Solver — exploratory data analysis.

Loads the raw train/test data and looks at the answer distribution, option
lengths, and duplicate/question-style patterns. Split out from the original
`final/dl-23f2004742-notebook-t22026.ipynb` (now archived at
`notebooks/_archive/`).


In [ ]:
import os, re, gc, math, random, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Config - CPU only
# ---------------------------------------------------------------------------
BASE        = "../data"
TRAIN_PATH  = f"{BASE}/train.csv"
TEST_PATH   = f"{BASE}/test.csv"
OUTPUT_PATH = "../outputs/submission.csv"

OPTIONS  = ["A", "B", "C", "D", "E"]
SEED     = 42
VAL_SIZE = 0.20
N_FOLDS  = 5

MPNET_ID = "sentence-transformers/all-mpnet-base-v2"

# ---------------------------------------------------------------------------
# Weights & Biases - one run per model, so every model has a tracked run with
# comparable metrics (MAP@3, accuracy, macro F1, weighted F1).
# Wrapped so that a W&B failure can never abort the run or lose the submission.
# ---------------------------------------------------------------------------
WANDB_PROJECT = "23f2004742-t22026"
WANDB_ON = True

os.environ["WANDB_SILENT"] = "true"

try:
    import wandb
    _key = None
    try:
        from kaggle_secrets import UserSecretsClient
        _key = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        _key = os.environ.get("WANDB_API_KEY")

    if _key:
        wandb.login(key=_key)
        print(f"W&B ready -> project '{WANDB_PROJECT}'")
    else:
        # A bare wandb.login() waits on stdin, which would hang a
        # "Save & Run All" commit forever. Disable instead of risking that.
        WANDB_ON = False
        print("W&B key not found (add WANDB_API_KEY as a Kaggle secret).")
        print("Continuing without tracking; all metrics are still printed.")
except Exception as e:
    WANDB_ON = False
    print(f"W&B unavailable ({type(e).__name__}); metrics still printed locally")

# DeBERTa is kept as an experiment only. It is a 435M model: fine-tuning it on
# CPU is not practical, so it stays off unless a GPU is attached.
RUN_DEBERTA = torch.cuda.is_available()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
print(f"torch {torch.__version__} | device: {DEVICE}")
print(f"DeBERTa experiment: {'ON' if RUN_DEBERTA else 'OFF (no GPU - expected on CPU)'}")


## 1. Load data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {train_df.shape}")
print(f"Test  : {test_df.shape}")

counts = train_df["answer"].value_counts().reindex(OPTIONS)
probs  = counts / counts.sum()
order  = list(probs.sort_values(ascending=False).index)

PRIOR_MAP3 = probs[order[0]] + probs[order[1]] / 2 + probs[order[2]] / 3
LABEL_LOGPRIOR = np.log(probs[OPTIONS].values.astype(np.float64))

print("\nAnswer distribution:")
for o in order:
    print(f"  {o}  {counts[o]:>4}  ({probs[o]:.1%})")
print(f"\nRandom-ordering MAP@3        : 0.3667")
print(f"Always '{' '.join(order[:3])}' MAP@3          : {PRIOR_MAP3:.4f}   <- the bar to beat")

train_df.head(3)
